In [1]:
import pandas as pd
import soccerdata as sd

print("Starting scrape...", flush=True)

leagues = [
    "ENG-Premier League",
    "ESP-La Liga", 
    "ITA-Serie A",
    "GER-Bundesliga",
    "FRA-Ligue 1",
]

for league in leagues:
    print(f"Trying {league}...", flush=True)
    try:
        u = sd.Understat(leagues=[league], seasons=["2025"], no_cache=True)
        df = u.read_schedule().reset_index()
        print(f"  Got {len(df)} rows", flush=True)
    except Exception as e:
        print(f"  Error: {e}", flush=True)

print("Done.", flush=True)

[09/14/26 15:04:20] INFO     No custom team name replacements found. You can configure these in       ]8;id=2822392;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=2822393;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\rhkha\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=2822399;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=2822400;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\rhkha\soccerdata\config\league_dict.json.                                    

Starting scrape...
Trying ENG-Premier League...


                    INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=2822407;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2822408;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-09-14 15:04:20] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


...

  Got 380 rows
Trying GER-Bundesliga...


[09/14/26 15:04:29] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=2822433;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2822434;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  Got 306 rows
Trying FRA-Ligue 1...


[09/14/26 15:04:32] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=2822439;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2822440;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  Got 306 rows
Done.


In [2]:
import sqlite3
import pandas as pd
import soccerdata as sd
from pathlib import Path

DB_PATH    = Path(r"C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\FIFA-WORLDCUP-PREDICTION\v4_historical_data.sqlite")
NEW_SEASON = "2526"
SCRAPE_SEASON = "2025"

TARGET_LEAGUES = [
    "ENG-Premier League",
    "ESP-La Liga",
    "ITA-Serie A",
    "GER-Bundesliga",
    "FRA-Ligue 1",
]

COLUMN_MAP = {
    "home_team"  : ["home_team", "home"],
    "away_team"  : ["away_team", "away"],
    "home_goals" : ["home_goals", "home_goal", "score_home"],
    "away_goals" : ["away_goals", "away_goal", "score_away"],
    "home_xg"    : ["home_xg", "xg_home", "xgh"],
    "away_xg"    : ["away_xg", "xg_away", "xga"],
    "is_finished": ["is_result", "finished", "status"],
}

def resolve_column(df, candidates):
    for name in candidates:
        if name in df.columns:
            return name
    return None

# Check existing data
conn = sqlite3.connect(DB_PATH)
before = pd.read_sql("SELECT season, COUNT(*) as n FROM matches_xg GROUP BY season ORDER BY season", conn)
conn.close()
print("Existing database:")
print(before.to_string(index=False))
print()

# Scrape
all_dfs = []
for league in TARGET_LEAGUES:
    print(f"Scraping {league}...", flush=True)
    u = sd.Understat(leagues=[league], seasons=[SCRAPE_SEASON], no_cache=True)
    df_raw = u.read_schedule().reset_index()
    print(f"  {len(df_raw)} rows", flush=True)
    all_dfs.append(df_raw)

df = pd.concat(all_dfs, ignore_index=True)

# Resolve columns
rename = {}
for standard, aliases in COLUMN_MAP.items():
    found = resolve_column(df, aliases)
    if found:
        rename[found] = standard

df = df.rename(columns=rename)

# Filter to finished matches
if "is_finished" in df.columns:
    df = df[df["is_finished"] == True].copy()
else:
    df = df.dropna(subset=["home_xg", "away_xg"]).copy()

# Standardise
for std, aliases in [("league",["league"]),("season",["season"]),("date",["date","datetime"])]:
    if std not in df.columns:
        found = resolve_column(df, aliases)
        if found:
            df = df.rename(columns={found: std})

required = ["league","season","date","home_team","away_team","home_goals","away_goals","home_xg","away_xg"]
df_clean = df[required].copy()
df_clean["date"]       = pd.to_datetime(df_clean["date"], errors="coerce")
df_clean["home_goals"] = pd.to_numeric(df_clean["home_goals"], errors="coerce")
df_clean["away_goals"] = pd.to_numeric(df_clean["away_goals"], errors="coerce")
df_clean["home_xg"]    = pd.to_numeric(df_clean["home_xg"],    errors="coerce")
df_clean["away_xg"]    = pd.to_numeric(df_clean["away_xg"],    errors="coerce")
df_clean = df_clean.dropna()

# Override season label to our short code
df_clean["season"] = NEW_SEASON

print(f"\nFinished matches with real xG: {len(df_clean):,}")
print(df_clean.groupby(["league","season"]).size().reset_index(name="matches").to_string(index=False))

# Append to database
conn = sqlite3.connect(DB_PATH)
df_clean.to_sql("matches_xg", conn, if_exists="append", index=False)
after = pd.read_sql("SELECT season, COUNT(*) as n FROM matches_xg GROUP BY season ORDER BY season", conn)
conn.close()

print("\nDatabase after append:")
print(after.to_string(index=False))
print(f"\n✅ {NEW_SEASON} appended successfully.")

Existing database:
season    n
  2122 1826
  2223 1826
  2324 1752
  2425 1752
  2526 1752

Scraping ENG-Premier League...


[09/14/26 15:04:35] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=2822445;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2822446;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  380 rows
Scraping ESP-La Liga...


[09/14/26 15:04:39] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=2822451;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2822452;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  380 rows
Scraping ITA-Serie A...


...

  380 rows
Scraping GER-Bundesliga...


[09/14/26 15:04:45] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=2822463;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2822464;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  306 rows
Scraping FRA-Ligue 1...


[09/14/26 15:04:48] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=2822469;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2822470;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  306 rows

Finished matches with real xG: 1,752
            league season  matches
ENG-Premier League   2526      380
       ESP-La Liga   2526      380
       FRA-Ligue 1   2526      306
    GER-Bundesliga   2526      306
       ITA-Serie A   2526      380

Database after append:
season    n
  2122 1826
  2223 1826
  2324 1752
  2425 1752
  2526 3504

✅ 2526 appended successfully.


In [15]:
import json, numpy as np, math
from scipy.stats import poisson
from datetime import datetime, timezone

with open(r"C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\FIFA-WORLDCUP-PREDICTION\v4_backend\v4_priors.json") as f:
    priors = json.load(f)

pl        = priors["ENG-Premier League"]["teams"]
meta      = priors["ENG-Premier League"]["meta"]
rho       = meta["rho_draw_correction"]
GAMMA_CAL = meta["gamma_home_advantage"]
DP        = 0.10

ALIASES = {
    "Spurs":             "Tottenham Hotspur",
    "Nott'm Forest":     "Nottingham Forest",
    "Nottingham Forest": "Nottingham Forest",
    "Man Utd":           "Manchester United",
    "Man City":          "Manchester City",
    "Coventry City":     "Coventry City",
    "Wolves":            "Wolverhampton Wanderers",
    "West Ham":          "West Ham United",
    "Newcastle":         "Newcastle United",
    "Brighton":          "Brighton & Hove Albion",
    "Brentford":         "Brentford",
}

SEASON_START = datetime(2026, 8, 21, tzinfo=timezone.utc)

def get_p(name):
    key = ALIASES.get(name, name)
    if key in pl: return pl[key]
    alphas = [v["alpha"] for v in pl.values()]
    betas  = [v["beta"]  for v in pl.values()]
    return {"alpha": np.percentile(alphas, 25), "beta": np.percentile(betas, 75)}

def eff_gamma(date_str):
    d = datetime.fromisoformat(date_str).replace(tzinfo=timezone.utc)
    days  = (d - SEASON_START).days
    gw    = max(1, days // 7 + 1)
    FLOOR = 0.60   # never drop below 60% of calibrated home advantage
    GW_FULL = 6    # fully restored by GW6 not GW10
    decay = FLOOR + (1.0 - FLOOR) * min(gw / GW_FULL, 1.0)
    return 1.0 + (GAMMA_CAL - 1.0) * decay

def dc(home, away, gamma):
    h = get_p(home); a = get_p(away)
    lam = np.clip(h["alpha"] * a["beta"] * gamma, 1e-5, 15)
    mu  = np.clip(a["alpha"] * h["beta"],          1e-5, 15)
    N = 9
    j = np.outer(poisson.pmf(np.arange(N), lam),
                 poisson.pmf(np.arange(N), mu))
    j[0,0]*=max(1-lam*mu*rho,1e-5); j[1,0]*=max(1+mu*rho,1e-5)
    j[0,1]*=max(1+lam*rho,1e-5);    j[1,1]*=max(1-rho,1e-5)
    j /= j.sum()
    ph=float(np.tril(j,-1).sum()); pd=float(np.trace(j)); pa=float(np.triu(j,+1).sum())
    tr=DP/2; ph2=max(ph-tr,0); pa2=max(pa-tr,0); pd2=pd+DP
    t=ph2+pd2+pa2
    return ph2/t, pd2/t, pa2/t

# GW1: Aug 21-24  |  GW2: Aug 28-31  |  GW3/4: Sep 12-14
# Results from NBC Sports + your notes
RESULTS = [
    # ── GW1 ───────────────────────────────────────────────────────
    # Fill in GW1 results — Aug 21-24
    # From NBC search: season started Aug 15 on NBC but your league Aug 21
    # Paste the real GW1 results from your Bing link here:
    # ("2026-08-21", "HOME", "AWAY", "H/D/A"),
    ("2026-08-21", "Arsenal",        "West Ham",           "H"),
    ("2026-08-22", "Manchester City","Fulham",             "H"),
    ("2026-08-22", "Chelsea",        "Everton",            "H"),
    ("2026-08-22", "Liverpool",      "Tottenham Hotspur",  "H"),
    ("2026-08-23", "Brighton",       "Sunderland",         "H"),
    ("2026-08-23", "Hull City",      "Coventry City",      "H"),
    ("2026-08-24", "Brentford",      "Crystal Palace",     "D"),
    ("2026-08-24", "Leeds",          "Aston Villa",        "H"),
    ("2026-08-24", "Newcastle",      "Ipswich Town",       "H"),
    ("2026-08-24", "Manchester United","Nottingham Forest","H"),
    # ── GW2 ───────────────────────────────────────────────────────
    # Fill in GW2 results — Aug 28-31
    ("2026-08-28", "Arsenal",        "Leeds",              "H"),
    ("2026-08-29", "Manchester City","Chelsea",            "H"),
    ("2026-08-29", "Liverpool",      "Hull City",          "H"),
    ("2026-08-29", "Tottenham Hotspur","Brighton",         "D"),
    ("2026-08-30", "Everton",        "Manchester United",  "D"),
    ("2026-08-30", "West Ham",       "Brentford",          "H"),
    ("2026-08-30", "Sunderland",     "Fulham",             "H"),
    ("2026-08-31", "Crystal Palace", "Aston Villa",        "H"),
    ("2026-08-31", "Coventry City",  "Newcastle",          "A"),
    ("2026-08-31", "Ipswich Town",   "Nottingham Forest",  "H"),

    # ── GW3/4: Sep 12-14 (what we already know) ───────────────────
    ("2026-09-12", "Aston Villa",    "Nottingham Forest", "A"),
    ("2026-09-12", "Bournemouth",    "Brentford",         "D"),
    ("2026-09-12", "Chelsea",        "Hull City",         "D"),
    ("2026-09-12", "Crystal Palace", "Ipswich Town",      "A"),
    ("2026-09-12", "Liverpool",      "Fulham",            "D"),
    ("2026-09-12", "Spurs",          "Everton",           "D"),
    ("2026-09-12", "Sunderland",     "Arsenal",           "A"),
    ("2026-09-12", "Coventry City",  "Brighton",          "A"),
    ("2026-09-12", "Man Utd",        "Man City",          "A"),
    ("2026-09-14", "Leeds",          "Newcastle",         "D"),
]

# ── Run comparison ──────────────────────────────────────────────
gw_groups = {
    "GW1 (Aug 21-24)": ("2026-08-21", "2026-08-25"),
    "GW2 (Aug 28-31)": ("2026-08-28", "2026-09-01"),
    "GW4 (Sep 12-14)": ("2026-09-12", "2026-09-15"),
}

print(f"{'Match':<32} {'GW':>3} {'Old H/D/A':>12} {'New H/D/A':>12} "
      f"{'Act':>3} {'Old':>4} {'New':>4}")
print("-"*80)

tot_oc=tot_nc=0; tot_oll=tot_nll=0; tot_n=0

for gw_label, (start, end) in gw_groups.items():
    games = [(d,h,a,r) for d,h,a,r in RESULTS if start<=d<end]
    if not games:
        print(f"\n{gw_label} — NO DATA YET")
        continue
    g_new = eff_gamma(start)
    gw_num = max(1, (datetime.fromisoformat(start).replace(tzinfo=timezone.utc)
                     - SEASON_START).days // 7 + 1)
    print(f"\n{gw_label} — gamma_old={GAMMA_CAL:.3f}  gamma_new={g_new:.3f}  "
          f"(home adv {(g_new-1)/(GAMMA_CAL-1)*100:.0f}% restored)")

    oc=nc=0; oll=nll=0
    for date,home,away,actual in games:
        ph_o,pd_o,pa_o = dc(home,away,GAMMA_CAL)
        ph_n,pd_n,pa_n = dc(home,away,g_new)
        pred_o = "H" if ph_o>pd_o and ph_o>pa_o else ("D" if pd_o>pa_o else "A")
        pred_n = "H" if ph_n>pd_n and ph_n>pa_n else ("D" if pd_n>pa_n else "A")
        ok_o = pred_o==actual; ok_n = pred_n==actual
        if ok_o: oc+=1
        if ok_n: nc+=1
        p_o={"H":ph_o,"D":pd_o,"A":pa_o}[actual]
        p_n={"H":ph_n,"D":pd_n,"A":pa_n}[actual]
        oll+=-math.log(max(p_o,1e-6))
        nll+=-math.log(max(p_n,1e-6))
        o_str=f"{ph_o*100:.0f}/{pd_o*100:.0f}/{pa_o*100:.0f}"
        n_str=f"{ph_n*100:.0f}/{pd_n*100:.0f}/{pa_n*100:.0f}"
        print(f"  {home} vs {away:<20} GW{gw_num} {o_str:>12} {n_str:>12} "
              f"{actual:>3} {'✓' if ok_o else '✗':>4} {'✓' if ok_n else '✗':>4}")

    n=len(games)
    print(f"  → Accuracy: Old {oc}/{n}={oc/n*100:.0f}%  New {nc}/{n}={nc/n*100:.0f}%  "
          f"| Log-loss: Old {oll/n:.3f}  New {nll/n:.3f}  "
          f"{'(New better)' if nll<oll else '(Old better)'}")
    tot_oc+=oc; tot_nc+=nc; tot_oll+=oll; tot_nll+=nll; tot_n+=n

if tot_n:
    print(f"\n{'='*80}")
    print(f"TOTAL ({tot_n} games)")
    print(f"  Accuracy:  Old {tot_oc}/{tot_n}={tot_oc/tot_n*100:.0f}%  "
          f"New {tot_nc}/{tot_n}={tot_nc/tot_n*100:.0f}%")
    print(f"  Log-loss:  Old {tot_oll/tot_n:.3f}  New {tot_nll/tot_n:.3f}  "
          f"{'New better ✓' if tot_nll<tot_oll else 'Old better'}")

Match                             GW    Old H/D/A    New H/D/A Act  Old  New
--------------------------------------------------------------------------------

GW1 (Aug 21-24) — gamma_old=1.236  gamma_new=1.157  (home adv 67% restored)
  Arsenal vs West Ham             GW1      73/24/3      71/26/4   H    ✓    ✓
  Manchester City vs Fulham               GW1     61/29/10     58/31/12   H    ✓    ✓
  Chelsea vs Everton              GW1     52/32/16     49/33/18   H    ✓    ✓
  Liverpool vs Tottenham Hotspur    GW1      78/21/1      75/22/2   H    ✓    ✓
  Brighton vs Sunderland           GW1     39/35/26     37/36/28   H    ✓    ✓
  Hull City vs Coventry City        GW1     39/35/26     37/36/28   H    ✓    ✓
  Brentford vs Crystal Palace       GW1     41/34/25     38/34/27   D    ✗    ✗
  Leeds vs Aston Villa          GW1     25/34/41     23/34/43   H    ✗    ✗
  Newcastle vs Ipswich Town         GW1      67/26/7      64/27/8   H    ✓    ✓
  Manchester United vs Nottingham Forest    GW1 

In [ ]:
# Run in notebook
import sqlite3

DB_PATH = r"C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\FIFA-WORLDCUP-PREDICTION\v4_web\data\predictions.db"
conn = sqlite3.connect(DB_PATH)

# Fix the score, result and correctness for Leeds vs Newcastle
# Leeds won 4-1 — predicted Away (Newcastle) → WRONG
conn.execute("""
    UPDATE predictions SET
        actual_result    = 'H',
        actual_hg        = 4,
        actual_ag        = 1,
        result_logged_at = datetime('now'),
        pre_correct      = 0,
        adj_correct      = 0
    WHERE home_team = 'Leeds' AND away_team = 'Newcastle'
      AND kickoff_utc LIKE '2026-09-14%'
""")
print("Rows updated:", conn.total_changes)

# Also fix the adj odds — should match DC prior not garbage values
conn.execute("""
    UPDATE predictions SET
        adj_home = pre_dc_home,
        adj_draw = pre_dc_draw,
        adj_away = pre_dc_away,
        adj_lam  = pre_dc_lam,
        adj_mu   = pre_dc_mu
    WHERE home_team = 'Leeds' AND away_team = 'Newcastle'
      AND kickoff_utc LIKE '2026-09-14%'
""")
print("Adj fixed:", conn.total_changes)

conn.commit()

# Verify
row = conn.execute("""
    SELECT home_team, away_team, actual_result, actual_hg, actual_ag,
           pre_dc_home, pre_dc_draw, pre_dc_away,
           adj_home, pre_correct
    FROM predictions
    WHERE home_team = 'Leeds' AND away_team = 'Newcastle'
""").fetchone()
print(dict(zip(
    [
        "home_team", "away_team", "actual_result", "actual_hg", "actual_ag",
        "pre_dc_home", "pre_dc_draw", "pre_dc_away", "adj_home", "pre_correct"
    ],
    row
)))
conn.close()

Rows updated: 1
Adj fixed: 2


ValueError: dictionary update sequence element #0 has length 5; 2 is required